# FreshCart ELT Pipeline

## Data Engineering Assignment – Part B

This notebook documents the FreshCart Extract, Load and Transform (ELT) pipeline.

**Implementation disclosure:** This project uses a local DuckDB database. The SQL transformations are adapted for DuckDB and are not presented as Snowflake execution evidence.

### Pipeline architecture

Extract → Landing → Bronze → Silver / Quarantine → Gold

The pipeline processes Web, Mobile, CRM, FX and Pincode source data.


## 1. Import Libraries and Connect to the Database

In [6]:

from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

# Locate the project root containing the src folder
PROJECT_ROOT = Path.cwd()

while PROJECT_ROOT != PROJECT_ROOT.parent:
    if (PROJECT_ROOT / "src").is_dir():
        break
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "src").is_dir():
    raise FileNotFoundError("Could not locate the project's src folder.")

# Add project root to Python import path
sys.path.insert(0, str(PROJECT_ROOT))

from src.db import get_connection

con = get_connection()

print("Project root:", PROJECT_ROOT)
print("Connected to FreshCart DuckDB database successfully.")


Project root: c:\Users\hp\Desktop\freshcart-pipelines
Connected to FreshCart DuckDB database successfully.


## 2. Extract Source Data

In [7]:
print("Using previously extracted FreshCart landing files.")
print("No new extraction run created.")

Using previously extracted FreshCart landing files.
No new extraction run created.


## 3. Load Extracted Data into Bronze

In [8]:
bronze_count = con.execute(
    "SELECT COUNT(*) FROM bronze.orders_raw"
).fetchone()[0]

print("Using existing Bronze data.")
print("Bronze order records:", bronze_count)

Using existing Bronze data.
Bronze order records: 164


## 4. Execute SQL Transformations

In [9]:
from src.transform import run_transformations

run_transformations()

print("All SQL transformation stages completed.")


Running 00_setup.sql...
Running 10_orders_clean.sql...
Running 20_orders_dedup.sql...
Running 30_customers_scd2.sql...
Running 31_fx_daily.sql...
Running 32_pincode_geo.sql...
Running 40_gold.sql...
All transformation stages completed.
All SQL transformation stages completed.


## 5. Inspect Silver and Quarantine Layers

In [10]:
print("Silver cleaned orders")
display(con.execute(
    "SELECT * FROM silver.orders_clean ORDER BY order_id"
).df())

print("Quarantined records")
display(con.execute(
    "SELECT * FROM quarantine.orders_rejected"
).df())


Silver cleaned orders


,ingest_seq,source,run_id,order_id,customer_email,customer_name,order_date,amount,currency,pincode,updated_at
0,368,web,20261112_125327,ORD-1001,priya.r@gmail.com,Priya r,2026-09-01,1250.0,INR,600041,2026-09-01 10:02:00
1,363,mobile,20261112_125327,ORD-1002,arjun.k@yahoo.in,Arjun k,2026-09-01,42.5,USD,560034,2026-09-01 11:15:00
2,370,web,20261112_125327,ORD-1002,arjun.k@yahoo.in,Arjun k,2026-09-01,42.5,USD,560034,2026-09-01 11:15:00
3,364,mobile,20261112_125327,ORD-1003,meera@outlook.com,Meera s,2026-09-02,2100.0,INR,400050,2026-09-02 18:40:00
4,369,web,20261112_125327,ORD-1003,meera@outlook.com,Meera s,2026-09-02,NaN,INR,400050,2026-09-02 09:10:00
5,372,web,20261112_125327,ORD-1005,priya.r@gmail.com,Priya r.,2026-09-03,899.0,INR,600041,2026-09-03 16:45:00


Quarantined records


,ingest_seq,source,run_id,order_id,rejection_reasons,record_status
0,373,web,20261112_125327,ORD-1006,invalid_email|unsupported_currency|invalid_pin...,REJECTED
1,371,web,20261112_125327,ORD-1004,non_positive_amount,REJECTED


## 6. Order Deduplication

In [11]:
dedup_df = con.execute("""
SELECT *
FROM silver.orders_deduplicated
ORDER BY order_id
""").df()

display(dedup_df)
print("Deduplicated order count:", len(dedup_df))


,ingest_seq,source,run_id,order_id,customer_email,customer_name,order_date,amount,currency,pincode,updated_at
0,368,web,20261112_125327,ORD-1001,priya.r@gmail.com,Priya r,2026-09-01,1250.0,INR,600041,2026-09-01 10:02:00
1,370,web,20261112_125327,ORD-1002,arjun.k@yahoo.in,Arjun k,2026-09-01,42.5,USD,560034,2026-09-01 11:15:00
2,364,mobile,20261112_125327,ORD-1003,meera@outlook.com,Meera s,2026-09-02,2100.0,INR,400050,2026-09-02 18:40:00
3,372,web,20261112_125327,ORD-1005,priya.r@gmail.com,Priya r.,2026-09-03,899.0,INR,600041,2026-09-03 16:45:00


Deduplicated order count: 4


## 7. Customer SCD Type 2 History

In [14]:
customer_history = con.execute(
    """
    SELECT *
    FROM silver.customer_tier_history
    ORDER BY customer_id, valid_from
    """
).df()

display(customer_history)

,customer_id,email,tier,valid_from,valid_to,is_current
0,CUST-01,priya.r@gmail.com,Silver,2023-04-10,2026-09-02,False
1,CUST-01,priya.r@gmail.com,Gold,2026-09-03,NaT,True
2,CUST-02,arjun.k@yahoo.in,Silver,2024-01-15,2026-08-14,False
3,CUST-02,arjun.k@yahoo.in,Gold,2026-08-15,NaT,True
4,CUST-03,meera@outlook.com,Platinum,2022-11-01,NaT,True


## 8. FX and Pincode Reference Data

In [19]:
print("Daily foreign exchange rates")

fx_data = con.execute(
    "SELECT * FROM silver.fx_rates"
).df()

display(fx_data)

print("Pincode geographic reference data")

pincode_data = con.execute(
    "SELECT * FROM silver.pincode_geo"
).df()

display(pincode_data)

Daily foreign exchange rates


,rate_date,currency,rate_to_inr
0,2026-09-01,USD,83.20
1,2026-09-02,USD,83.35
2,2026-09-03,USD,83.10
3,2026-09-01,INR,1.00
4,2026-09-02,INR,1.00
5,2026-09-03,INR,1.00


Pincode geographic reference data


,pincode,city,state
0,400050,Mumbai,Maharashtra
1,560034,Bengaluru,Karnataka
2,600041,Chennai,Tamil Nadu


## 9. Gold Analytical Layer

In [16]:
gold_df = con.execute("""
SELECT *
FROM gold.orders
ORDER BY order_id
""").df()

display(gold_df)


,order_id,customer_email,customer_name,order_date,amount,currency,pincode,updated_at,customer_tier,rate_to_inr,city,state,amount_inr,loyalty_points
0,ORD-1001,priya.r@gmail.com,Priya r,2026-09-01,1250.0,INR,600041,2026-09-01 10:02:00,Silver,1.0,Chennai,Tamil Nadu,1250.0,12
1,ORD-1002,arjun.k@yahoo.in,Arjun k,2026-09-01,42.5,USD,560034,2026-09-01 11:15:00,Gold,83.2,Bengaluru,Karnataka,3536.0,70
2,ORD-1003,meera@outlook.com,Meera s,2026-09-02,2100.0,INR,400050,2026-09-02 18:40:00,Platinum,1.0,Mumbai,Maharashtra,2100.0,63
3,ORD-1005,priya.r@gmail.com,Priya r.,2026-09-03,899.0,INR,600041,2026-09-03 16:45:00,Gold,1.0,Chennai,Tamil Nadu,899.0,16


## 10. Revenue and Loyalty Summary

In [17]:
gold_summary = con.execute("""
SELECT
    COUNT(*) AS total_gold_orders,
    ROUND(SUM(amount_inr), 2) AS total_revenue_inr,
    SUM(loyalty_points) AS total_loyalty_points
FROM gold.orders
""").df()

display(gold_summary)


,total_gold_orders,total_revenue_inr,total_loyalty_points
0,4,7785.0,161.0


## 11. Data Quality Checks

In [20]:
from src.quality_checks import quality_checks

quality_passed = quality_checks()

print("All data quality checks passed:", quality_passed)


\n=== FRESHCART DATA QUALITY CHECKS ===
Check 1: Gold order IDs must be unique | Failed rows: 0 | PASS
Check 2: Gold order IDs must not be NULL | Failed rows: 0 | PASS
Check 3: Gold amounts must be positive | Failed rows: 0 | PASS
Check 4: Gold customer tiers must not be NULL | Failed rows: 0 | PASS
Check 5: Gold loyalty points must not be negative | Failed rows: 0 | PASS
Check 6: Gold INR amount must exist when source amount exists | Failed rows: 0 | PASS
\nOverall: ALL CHECKS PASSED
All data quality checks passed: True


## 12. Final Pipeline Summary

In [21]:
summary_queries = {
    "Bronze orders": "SELECT COUNT(*) FROM bronze.orders_raw",
    "Silver cleaned": "SELECT COUNT(*) FROM silver.orders_clean",
    "Quarantine": "SELECT COUNT(*) FROM quarantine.orders_rejected",
    "Deduplicated": "SELECT COUNT(*) FROM silver.orders_deduplicated",
    "Gold orders": "SELECT COUNT(*) FROM gold.orders",
    "Gold revenue (INR)": "SELECT ROUND(SUM(amount_inr), 2) FROM gold.orders",
    "Loyalty points": "SELECT SUM(loyalty_points) FROM gold.orders"
}

summary = []

for metric, query in summary_queries.items():
    value = con.execute(query).fetchone()[0]
    summary.append({"Metric": metric, "Result": value})

summary_df = pd.DataFrame(summary)
display(summary_df)

con.close()
print("FreshCart ELT notebook execution completed.")


,Metric,Result
0,Bronze orders,164
1,Silver cleaned,6
2,Quarantine,2
3,Deduplicated,4
4,Gold orders,4
5,Gold revenue (INR),7785.00
6,Loyalty points,161


FreshCart ELT notebook execution completed.
